# AI Agent vs. Human Detection: CUDA-Accelerated Evaluation

## 1. Introduction

This notebook evaluates pre-trained machine learning models using pre-split training, testing, and evaluation datasets. It is optimized for CUDA to utilize GPU acceleration.

## 2. Setup and CUDA Check

In [2]:
!pip install pytorch-tabnet2 rtdl-revisiting-models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 19.1 MB/s eta 0:00:00


In [17]:
import pandas as pd
import numpy as np
import pickle
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from pytorch_tabnet import TabNetRegressor
from rtdl_revisiting_models import FTTransformer
from sklearn.metrics import f1_score, accuracy_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
# from sklearn.datasets import make_regression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, mean_squared_error, r2_score

# Check for CUDA availability
device = torch.device("cuda") if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == 'cuda': # Corrected: Direct comparison as device might be a string 'cpu'
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda


## 3. Loading Pre-split Data

In [5]:
# def load_pickle(file_path):
#     with open(file_path, 'rb') as f:
#         return pickle.load(f)

# # Replace with your actual pre-split pickle file paths
# train_data = load_pickle('processed_v1_features_train.pkl')
# test_data = load_pickle('processed_v1_features_test.pkl')
# eval_data = load_pickle('processed_v1_features_val.pkl')

train_data = pd.read_pickle('processed_v1_features_train.pkl')
test_data = pd.read_pickle('processed_v1_features_test.pkl')
eval_data = pd.read_pickle('processed_v1_features_val.pkl')

# Check dimensions of pickle files
print(f"train_data type: {type(train_data)}")
print(f"train_data shape: {train_data.shape if hasattr(train_data, 'shape') else 'N/A'}")
print(f"train_data ndim: {train_data.ndim if hasattr(train_data, 'ndim') else 'N/A'}")
print(f"train_data dtype: {train_data.dtype if hasattr(train_data, 'dtype') else 'N/A'}")

train_data type: <class 'numpy.ndarray'>
train_data shape: (24514, 773)
train_data ndim: 2
train_data dtype: float64


In [6]:
X_train, y_train = train_data[:, :-1], train_data[:, -1]
X_test, y_test = test_data[:, :-1], test_data[:, -1]
X_eval, y_eval = eval_data[:, :-1], eval_data[:, -1]

## 4. Regression Model Training

In [7]:
# 1. Initialize the Random Forest Regression
# n_estimators: number of trees
# n_jobs=-1: uses all your CPU cores for faster training
rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Train the model
print("Training the Random Forest Regressor...")
rf_regressor.fit(X_train, y_train)

# 3. Evaluate on the Validation (Eval) set
y_pred_eval = rf_regressor.predict(X_eval)

# For regression, we use MSE and R-squared instead of Accuracy
mse = mean_squared_error(y_eval, y_pred_eval)
r2 = r2_score(y_eval, y_pred_eval)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared Score: {r2:.4f}")

Training the Random Forest Regressor...
Mean Squared Error: 0.0097
R-squared Score: 0.6272


A random forest regression model was used to

In [15]:
tabnet_regressor = TabNetRegressor(
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
)

# 2. Train the model
print("Training the TabNet Regressor...")
tabnet_regressor.fit(
    X_train = X_train, # X_train should not be reshaped to (-1,1) as it's already (N_samples, N_features)
    y_train = y_train.reshape(-1, 1), # Reshape y_train to (N_samples, 1)
    eval_set=[(X_eval, y_eval.reshape(-1, 1))],
    max_epochs=25 # Reshape y_eval to (N_samples, 1) for consistency
)

# 3. Evaluate on validation set
y_pred_eval_tabnet = tabnet_regressor.predict(X_eval)
mse_tabnet = mean_squared_error(y_eval, y_pred_eval_tabnet)
r2_tabnet = r2_score(y_eval, y_pred_eval_tabnet)

print(f"TabNet Mean Squared Error: {mse_tabnet:.4f}")
print(f"TabNet R-squared Score: {r2_tabnet:.4f}")

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/tab_models/tab_reg.py:22: UserWarning: Device used : cuda
  super(TabNetRegressor, self).__post_init__()


Training the TabNet Regressor...
epoch 0  | loss: 0.15021 | val_0_mse: 0.03314 |  0:00:00s
epoch 1  | loss: 0.02607 | val_0_mse: 0.02732 |  0:00:00s
epoch 2  | loss: 0.02223 | val_0_mse: 0.0242  |  0:00:00s
epoch 3  | loss: 0.02083 | val_0_mse: 0.02156 |  0:00:00s
epoch 4  | loss: 0.01948 | val_0_mse: 0.02066 |  0:00:00s
epoch 5  | loss: 0.01876 | val_0_mse: 0.01898 |  0:00:01s
epoch 6  | loss: 0.01835 | val_0_mse: 0.01823 |  0:00:01s
epoch 7  | loss: 0.01801 | val_0_mse: 0.01789 |  0:00:01s
epoch 8  | loss: 0.0175  | val_0_mse: 0.01686 |  0:00:01s
epoch 9  | loss: 0.01715 | val_0_mse: 0.01554 |  0:00:01s
epoch 10 | loss: 0.01649 | val_0_mse: 0.01523 |  0:00:01s
epoch 11 | loss: 0.01598 | val_0_mse: 0.01494 |  0:00:02s
epoch 12 | loss: 0.01575 | val_0_mse: 0.01429 |  0:00:02s
epoch 13 | loss: 0.01519 | val_0_mse: 0.01383 |  0:00:02s
epoch 14 | loss: 0.01457 | val_0_mse: 0.01311 |  0:00:02s
epoch 15 | loss: 0.01408 | val_0_mse: 0.01281 |  0:00:02s
epoch 16 | loss: 0.01353 | val_0_mse: 0

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks/container.py:113: UserWarning: Best weights from best epoch are automatically used!
  callback.on_train_end(logs)


In [19]:
# 1. Initialise the Multi Layer Perceptron algorithm
mlp = MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)

# 2. Train the model
print("Training the Multi Layer Perceptron...")
mlp.fit(X_train, y_train)

# 3. Evaluate Validation set
y_pred_eval = mlp.predict(X_eval)
mse = mean_squared_error(y_eval, y_pred_eval)
r2 = r2_score(y_eval, y_pred_eval)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared Score: {r2:.4f}")

Training the Multi Layer Perceptron...
Mean Squared Error: 0.0028
R-squared Score: 0.8932


In [ ]:
# cross validation to check score
from sklearn.model_selection import cross_val_score


In [ ]:
# Check the structure of loaded data
print(f"train_data type: {type(train_data)}")
print(f"train_data keys/structure: {train_data.keys() if isinstance(train_data, dict) else train_data if isinstance(train_data, tuple) else 'unknown'}")

# If it's a dict, show available keys
if isinstance(train_data, dict):
    print(f"Available keys: {list(train_data.keys())}")
    # Check the first key's contents
    first_key = list(train_data.keys())[0]
    print(f"Sample of first key: {type(train_data[first_key])}")

## 4. Loading and Running Models on CUDA

In [ ]:
# Note: For Scikit-learn models (Random Forest, etc.), CUDA is typically used via libraries like cuML or by using specific GPU-enabled implementations (e.g., XGBoost, CatBoost).
# If the models are PyTorch-based, we move them to the device.

def run_model_on_cuda(model_path, X_data):
    model = load_pickle(model_path)

    # If the model is a PyTorch model
    if hasattr(model, 'to'):
        model = model.to(device)
        X_tensor = torch.tensor(X_data.values if hasattr(X_data, 'values') else X_data).float().to(device)
        with torch.no_grad():
            preds = model(X_tensor)
            return preds.cpu().numpy()

    # If the model is an XGBoost/CatBoost model with GPU support
    # (These usually handle device placement internally if configured)
    return model.predict(X_data)

# Paths to your model pickles
model_paths = {
    'Model_1': 'model_1.pkl',
    'Model_2': 'model_2.pkl'
}

results = []
for name, path in model_paths.items():
    print(f"Evaluating {name}...")
    y_pred = run_model_on_cuda(path, X_eval)

    # Handle probabilistic outputs if necessary
    if len(y_pred.shape) > 1 and y_pred.shape[1] > 1:
        y_pred = np.argmax(y_pred, axis=1)

    f1 = f1_score(y_eval, y_pred)
    results.append({'Model': name, 'F1 Score': f1})

eval_df = pd.DataFrame(results)
print(eval_df)
```

## 5. Final Comparison and Visualization

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=eval_df, x='Model', y='F1 Score')
plt.title('Model F1 Score Comparison (CUDA Accelerated)')
plt.show()



The error ImportError: cannot import name 'TabnetRegressor' from 'pytorch_tabnet' in cell 30637f64 indicates that the class name TabnetRegressor is incorrect. In the pytorch_tabnet library, the correct class name for the regressor is TabNetRegressor (with a capital 'N'). I will fix this in the code.


Please explain this error:

The error ImportError: cannot import name 'SparsePredictDataset' from 'pytorch_tabnet.utils' means that the pytorch_tabnet library, when trying to import TabnetRegressor, couldn't find an internal component it needs. This is likely due to the pytorch-tabnet2 package being installed, which may be incompatible. I'll correct the capitalization of TabnetRegressor to TabNetRegressor in the import statement, as this is the correct class name for the library. This might resolve the internal dependency. If the error persists, you might need to install the original pytorch-tabnet library instead of pytorch-tabnet2.


Please explain this error:

The error ImportError: cannot import name 'TabnetRegressor' from 'pytorch_tabnet' means that the TabnetRegressor class isn't directly exposed by the main pytorch_tabnet package. You need to import it from its specific submodule, which is usually pytorch_tabnet.tab_model, and use the correct capitalization: TabNetRegressor. I'll update the import statement in cell 30637f64 to reflect this.


Please explain this error:

The error AttributeError: 'str' object has no attribute 'type' means that when CUDA is not available, the device variable becomes the string 'cpu'. You can't access a .type attribute on a string. I'll correct the code to directly compare device to the string 'cuda' to fix this.

Great news! The previous fix for the AttributeError worked, and cell 30637f64 has executed successfully. It looks like your environment is currently using the CPU for computations. The next step is to load the pre-split data, but I noticed cell 5581df55 failed with a FileNotFoundError earlier. Would you like me to address that next?


Please explain this error:

The error "ValueError: too many values to unpack (expected 2)" happens because train_data, test_data, and eval_data are single arrays, but the code is trying to assign them to two variables at once. To fix this, I'll update the code to slice each array, separating the features (X) from the labels (y). I'm assuming the last column of each array represents the labels.


Please explain this error:

The error UnpicklingError: pickle data was truncated means that the processed_v1_features_train.pkl file is either corrupted or incomplete. The system tried to load it, but the file ended unexpectedly. You'll need to ensure that this pickle file, and potentially processed_v1_features_test.pkl and processed_v1_features_val.pkl, are correctly generated and present in the expected location. If there are previous steps in the notebook meant to create these files, please ensure they run without issues.


how do i fix the dimensions?
The error regarding dimension mismatch in the TabNet Regressor training means that the target arrays for training and evaluation need to have consistent shapes. I'll correct the code by ensuring that both y_train and y_eval are reshaped to (N, 1) (number of samples, 1 target), and that X_train is passed without an unnecessary reshape, as it's already in the correct (N_samples, N_features) format.

The TabNet Regressor has finished training on the GPU, which is great! However, the evaluation part of the code was incomplete. I'll now add the necessary lines to make predictions on the evaluation set and then calculate the Mean Squared Error and R-squared score for the TabNet model.


i also want to restrict it to 25 epochs

Update max_epochs to 25 in the TabNet fit method
Okay, I will update the max_epochs to 25 in the TabNet fit method.

The TabNet Regressor has finished training for 25 epochs. The Mean Squared Error on the evaluation set is 0.0091, and the R-squared Score is 0.6510. This indicates a good performance, with the model explaining about 65.1% of the variance in the target variable on the evaluation set.


is 65.1% the best accuracy, or is it the average of accuracies?
The R-squared Score of 65.1% for the TabNet Regressor is the best performance achieved on the evaluation set. During training, the model tracks its performance (in this case, Mean Squared Error) on the evaluation set. When it reports best_epoch = 23, it means the model performed best at that epoch, and the metrics (MSE and R-squared) are calculated using the model's weights from that specific best epoch.


Please explain this error:

The error ValueError: Unknown label type arises because you are using an MLPClassifier with y_train, which contains continuous numerical values, making it a regression problem. Classifiers are designed for discrete labels. To fix this, I will change MLPClassifier to MLPRegressor and update the evaluation metrics to Mean Squared Error and R-squared score, consistent with regression tasks.

The Multi-Layer Perceptron (MLP) model has been successfully trained as a regressor. It achieved a Mean Squared Error of 0.0028 and an R-squared Score of 0.8932 on the evaluation set. This indicates a strong performance, with the model explaining about 89.32% of the variance in the target variable, making it the best performing model so far compared to Random Forest and TabNet.


can you compare and explain the performance of the various models (random forest regression, tabnet, mlpregressor) for this use case? and how may i extract the significant features for extraction?
Certainly! Let's compare the performance of the Random Forest Regressor, TabNet Regressor, and MLP Regressor based on their Mean Squared Error (MSE) and R-squared (R2) scores:

Mean Squared Error (MSE): Measures the average squared difference between the estimated values and the actual value. Lower MSE indicates better model performance.
R-squared (R2) Score: Represents the proportion of the variance in the dependent variable that is predictable from the independent variables. A higher R-squared value indicates a better fit for the model.
Here's a summary of their performance:

Random Forest Regressor:
Mean Squared Error: 0.0097
R-squared Score: 0.6272
TabNet Regressor:
Mean Squared Error: 0.0091
R-squared Score: 0.6510
MLP Regressor:
Mean Squared Error: 0.0028
R-squared Score: 0.8932
Model Comparison:
From these results, the MLP Regressor clearly outperforms the other two models by a significant margin. It has the lowest MSE and the highest R-squared score, indicating that it explains a much larger proportion of the variance in the target variable and makes more accurate predictions.

The TabNet Regressor performs slightly better than the Random Forest Regressor in terms of both MSE and R-squared.

Feature Extraction:
Extracting significant features depends on the type of model you are using:

For Random Forest Regressor (rf_regressor): Tree-based models like Random Forest provide direct feature importance scores. You can access these using the feature_importances_ attribute.

import pandas as pd

# Assuming you have feature names (e.g., if you loaded from a DataFrame initially)
# If not, you might have to label them X0, X1, ..., Xn-1
# For this example, let's assume `X_train` columns correspond to features
# If X_train was originally a DataFrame, you'd use X_train.columns
# Since it's a numpy array, we'll create generic names.
feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]

importances = rf_regressor.feature_importances_
feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
print("Random Forest Feature Importances:")
print(feature_importance_df.head(10))
For TabNet Regressor (tabnet_regressor): TabNet models also provide a way to inspect feature importance, often through the feature_importances_ attribute after training.

import pandas as pd

feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]

tabnet_importances = tabnet_regressor.feature_importances_
# Ensure the importances array is 1D if it's not already
if len(tabnet_importances.shape) > 1:
    tabnet_importances = tabnet_importances.flatten()

tabnet_feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': tabnet_importances})
tabnet_feature_importance_df = tabnet_feature_importance_df.sort_values(by='importance', ascending=False)
print("\nTabNet Feature Importances:")
print(tabnet_feature_importance_df.head(10))
For MLP Regressor (mlp): Neural networks like MLP do not have a straightforward feature_importances_ attribute like tree-based models. However, you can use model-agnostic techniques like Permutation Importance to determine feature significance. This involves shuffling a single feature's values and observing how much the model's performance (e.g., MSE) decreases. A larger decrease indicates a more important feature.

from sklearn.inspection import permutation_importance

feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]

# For regression, we typically look at how much the error increases
result = permutation_importance(mlp, X_eval, y_eval, n_repeats=10, random_state=42, n_jobs=-1, scoring='neg_mean_squared_error')
# Note: scoring='neg_mean_squared_error' means higher values are better, so we sort by mean of the importance_values

mlp_importance_df = pd.DataFrame({'feature': feature_names, 'importance_mean': result.importances_mean, 'importance_std': result.importances_std})
mlp_importance_df = mlp_importance_df.sort_values(by='importance_mean', ascending=False)
print("\nMLP Permutation Importances (based on Mean Squared Error increase):")
print(mlp_importance_df.head(10))
Would you like me to generate code cells to extract and display the top significant features for any of these models?